# 股票技术指标计算与可视化 - 代码复现手册

本Notebook包含所有技术指标的计算代码实现，便于学习和复现。

## 目录
1. 数据加载
2. 移动平均线 (MA)
3. 相对强弱指标 (RSI)
4. MACD指标
5. 布林带 (Bollinger Bands)
6. 平均真实波幅 (ATR)
7. DMI/ADX指标
8. KDJ指标
9. VWAP指标
10. 综合示例：参数调节效果对比

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("库导入成功！")

## 1. 数据加载

从CSV文件加载股票数据，或进行模拟数据生成。

In [ ]:
def load_stock_data(file_path):
    """
    加载股票数据
    
    参数:
        file_path: CSV文件路径
    
    返回:
        DataFrame，包含ts_code, trade_date, open, high, low, close, vol, amount等字段
    """
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # 转换日期格式
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    
    # 按日期排序
    df = df.sort_values('trade_date').reset_index(drop=True)
    
    return df

# 示例：加载数据（请修改为实际文件路径）
# df = load_stock_data('data/ningde_times_300750_daily.csv')
# print(f"数据加载成功！共 {len(df)} 条记录")
# df.head()

In [ ]:
def generate_sample_data(n_days=500):
    """
    生成模拟股票数据（用于测试）
    
    参数:
        n_days: 生成数据的天数
    
    返回:
        DataFrame，包含模拟的OHLC数据
    """
    np.random.seed(42)
    
    dates = pd.date_range(end=datetime.today(), periods=n_days, freq='B')
    
    # 生成价格数据（随机游走）
    returns = np.random.normal(0.0005, 0.02, n_days)
    prices = 100 * np.exp(np.cumsum(returns))
    
    data = []
    for i, date in enumerate(dates):
        close = prices[i]
        high = close * (1 + abs(np.random.normal(0, 0.01)))
        low = close * (1 - abs(np.random.normal(0, 0.01)))
        open_price = close * (1 + np.random.normal(0, 0.005))
        open_price = max(min(open_price, high), low)  # 确保开盘价在高低价之间
        
        vol = np.random.randint(100000, 500000)
        amount = vol * close / 100  # 近似成交额
        
        data.append({
            'ts_code': '000001.SZ',
            'trade_date': date,
            'open': round(open_price, 2),
            'high': round(high, 2),
            'low': round(low, 2),
            'close': round(close, 2),
            'vol': vol,
            'amount': round(amount, 2)
        })
    
    df = pd.DataFrame(data)
    return df

# 生成模拟数据
df = generate_sample_data(500)
print("模拟数据生成成功！")
print(f"数据期间: {df['trade_date'].min().date()} 至 {df['trade_date'].max().date()}")
df.head()

## 2. 移动平均线 (MA)

移动平均线是最基础的技术指标，用于平滑价格波动，识别趋势方向。

In [ ]:
def calculate_ma(df, periods=[5, 10, 20, 60]):
    """
    计算移动平均线
    
    参数:
        df: 股票数据DataFrame
        periods: MA周期列表，默认[5, 10, 20, 60]
    
    返回:
        DataFrame，新增MA列（如ma5, ma10, ma20, ma60）
    """
    result = df.copy()
    
    for period in periods:
        col_name = f'ma{period}'
        result[col_name] = result['close'].rolling(window=period).mean()
    
    return result

# 示例：计算MA
df_with_ma = calculate_ma(df, periods=[5, 10, 20])
df_with_ma[['trade_date', 'close', 'ma5', 'ma10', 'ma20']].tail(10)

In [ ]:
def plot_ma(df, ma_periods=[5, 10, 20]):
    """
    绘制K线图和移动平均线
    
    参数:
        df: 包含MA列的DataFrame
        ma_periods: 要绘制的MA周期
    """
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # 绘制收盘价
    ax.plot(df['trade_date'], df['close'], color='black', linewidth=1, label='收盘价')
    
    # 绘制MA线
    colors = ['red', 'blue', 'green', 'purple']
    for i, period in enumerate(ma_periods):
        col_name = f'ma{period}'
        if col_name in df.columns:
            ax.plot(df['trade_date'], df[col_name], 
                   color=colors[i % len(colors)], 
                   linewidth=1.5, 
                   label=f'MA{period}')
    
    ax.set_title('移动平均线 (MA)', fontsize=14, fontweight='bold')
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('价格', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制MA图（显示最后100个交易日）
plot_ma(df_with_ma.tail(100), ma_periods=[5, 10, 20])

## 3. 相对强弱指标 (RSI)

RSI衡量价格变动的速度和幅度，用于识别超买（>70）和超卖（<30）区域。

In [ ]:
def calculate_rsi(df, period=14):
    """
    计算RSI指标
    
    参数:
        df: 股票数据DataFrame
        period: RSI周期，默认14
    
    返回:
        DataFrame，新增rsi列
    """
    result = df.copy()
    
    # 计算价格变化
    delta = result['close'].diff()
    
    # 分离上涨和下跌
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    # 计算平均上涨和平均下跌（使用指数移动平均）
    avg_gain = gain.ewm(com=period-1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period-1, min_periods=period).mean()
    
    # 计算RS和RSI
    rs = avg_gain / avg_loss
    result[f'rsi{period}'] = 100 - (100 / (1 + rs))
    
    return result

# 示例：计算RSI
df_with_rsi = calculate_rsi(df_with_ma, period=14)
df_with_rsi[['trade_date', 'close', 'rsi14']].tail(10)

In [ ]:
def plot_rsi(df, period=14, overbought=70, oversold=30):
    """
    绘制RSI指标图
    
    参数:
        df: 包含RSI列的DataFrame
        period: RSI周期
        overbought: 超买线，默认70
        oversold: 超卖线，默认30
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 上图：收盘价
    ax1.plot(df['trade_date'], df['close'], color='black', linewidth=1)
    ax1.set_ylabel('价格', fontsize=12)
    ax1.set_title('价格走势', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 下图：RSI
    col_name = f'rsi{period}'
    ax2.plot(df['trade_date'], df[col_name], color='blue', linewidth=1.5)
    ax2.axhline(y=overbought, color='red', linestyle='--', alpha=0.5, label='超买线')
    ax2.axhline(y=oversold, color='green', linestyle='--', alpha=0.5, label='超卖线')
    ax2.fill_between(df['trade_date'], overbought, 100, alpha=0.1, color='red')
    ax2.fill_between(df['trade_date'], 0, oversold, alpha=0.1, color='green')
    
    ax2.set_ylabel('RSI', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_title(f'RSI ({period})', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制RSI图
plot_rsi(df_with_rsi.tail(100), period=14)

## 4. MACD指标

MACD由快线（DIF）、慢线（DEA）和柱状图组成，用于识别趋势反转。

In [ ]:
def calculate_macd(df, fast_period=12, slow_period=26, signal_period=9):
    """
    计算MACD指标
    
    参数:
        df: 股票数据DataFrame
        fast_period: 快线周期，默认12
        slow_period: 慢线周期，默认26
        signal_period: 信号线周期，默认9
    
    返回:
        DataFrame，新增dif, dea, macd列
    """
    result = df.copy()
    
    # 计算快线和慢线（EMA）
    ema_fast = result['close'].ewm(span=fast_period, adjust=False).mean()
    ema_slow = result['close'].ewm(span=slow_period, adjust=False).mean()
    
    # 计算DIF（快线 - 慢线）
    result['dif'] = ema_fast - ema_slow
    
    # 计算DEA（DIF的EMA）
    result['dea'] = result['dif'].ewm(span=signal_period, adjust=False).mean()
    
    # 计算MACD柱状图（(DIF - DEA) * 2）
    result['macd'] = (result['dif'] - result['dea']) * 2
    
    return result

# 示例：计算MACD
df_with_macd = calculate_macd(df_with_rsi)
df_with_macd[['trade_date', 'close', 'dif', 'dea', 'macd']].tail(10)

In [ ]:
def plot_macd(df, fast_period=12, slow_period=26, signal_period=9):
    """
    绘制MACD指标图
    
    参数:
        df: 包含MACD列的DataFrame
        fast_period: 快线周期
        slow_period: 慢线周期
        signal_period: 信号线周期
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 上图：收盘价
    ax1.plot(df['trade_date'], df['close'], color='black', linewidth=1)
    ax1.set_ylabel('价格', fontsize=12)
    ax1.set_title('价格走势', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 下图：MACD
    ax2.plot(df['trade_date'], df['dif'], color='blue', linewidth=1.5, label='DIF')
    ax2.plot(df['trade_date'], df['dea'], color='red', linewidth=1.5, label='DEA')
    
    # 绘制MACD柱状图
    colors = ['red' if x < 0 else 'green' for x in df['macd']]
    ax2.bar(df['trade_date'], df['macd'], color=colors, alpha=0.5, label='MACD')
    
    ax2.set_ylabel('MACD', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_title(f'MACD ({fast_period},{slow_period},{signal_period})', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制MACD图
plot_macd(df_with_macd.tail(100))

## 5. 布林带 (Bollinger Bands)

布林带由中轨（MA）、上轨和下轨组成，用于衡量价格波动范围。

In [ ]:
def calculate_bollinger_bands(df, period=20, std_dev=2):
    """
    计算布林带
    
    参数:
        df: 股票数据DataFrame
        period: 周期，默认20
        std_dev: 标准差倍数，默认2
    
    返回:
        DataFrame，新增bb_middle, bb_upper, bb_lower列
    """
    result = df.copy()
    
    # 计算中轨（SMA）
    result['bb_middle'] = result['close'].rolling(window=period).mean()
    
    # 计算标准差
    std = result['close'].rolling(window=period).std()
    
    # 计算上轨和下轨
    result['bb_upper'] = result['bb_middle'] + (std * std_dev)
    result['bb_lower'] = result['bb_middle'] - (std * std_dev)
    
    return result

# 示例：计算布林带
df_with_bb = calculate_bollinger_bands(df_with_macd)
df_with_bb[['trade_date', 'close', 'bb_middle', 'bb_upper', 'bb_lower']].tail(10)

In [ ]:
def plot_bollinger_bands(df, period=20, std_dev=2):
    """
    绘制布林带
    
    参数:
        df: 包含布林带列的DataFrame
        period: 周期
        std_dev: 标准差倍数
    """
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # 绘制K线（简化：用收盘价线代替）
    ax.plot(df['trade_date'], df['close'], color='black', linewidth=1, label='收盘价')
    
    # 绘制布林带
    ax.plot(df['trade_date'], df['bb_middle'], color='blue', linewidth=1.5, label='中轨')
    ax.plot(df['trade_date'], df['bb_upper'], color='red', linewidth=1, linestyle='--', label='上轨')
    ax.plot(df['trade_date'], df['bb_lower'], color='green', linewidth=1, linestyle='--', label='下轨')
    
    # 填充上下轨之间的区域
    ax.fill_between(df['trade_date'], df['bb_upper'], df['bb_lower'], alpha=0.1, color='gray')
    
    ax.set_title(f'布林带 ({period}, {std_dev}倍标准差)', fontsize=14, fontweight='bold')
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('价格', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制布林带图
plot_bollinger_bands(df_with_bb.tail(100))

## 6. 平均真实波幅 (ATR)

ATR衡量价格波动的真实范围，用于设置止损位。

In [ ]:
def calculate_atr(df, period=14):
    """
    计算ATR指标
    
    参数:
        df: 股票数据DataFrame
        period: 周期，默认14
    
    返回:
        DataFrame，新增atr列
    """
    result = df.copy()
    
    # 计算真实波幅（TR）
    high_low = result['high'] - result['low']
    high_close = abs(result['high'] - result['close'].shift(1))
    low_close = abs(result['low'] - result['close'].shift(1))
    
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    
    # 计算ATR（使用指数移动平均）
    result[f'atr{period}'] = tr.ewm(com=period-1, min_periods=period).mean()
    
    return result

# 示例：计算ATR
df_with_atr = calculate_atr(df_with_bb)
df_with_atr[['trade_date', 'close', 'atr14']].tail(10)

In [ ]:
def plot_atr(df, period=14):
    """
    绘制ATR指标图
    
    参数:
        df: 包含ATR列的DataFrame
        period: 周期
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 上图：收盘价
    ax1.plot(df['trade_date'], df['close'], color='black', linewidth=1)
    ax1.set_ylabel('价格', fontsize=12)
    ax1.set_title('价格走势', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 下图：ATR
    col_name = f'atr{period}'
    ax2.plot(df['trade_date'], df[col_name], color='blue', linewidth=1.5)
    
    # 添加止损线示例（收盘价 - 2倍ATR）
    stop_loss = df['close'] - 2 * df[col_name]
    ax1.plot(df['trade_date'], stop_loss, color='red', linestyle=':', alpha=0.5, label='止损线 (close - 2*ATR)')
    ax1.legend(loc='best')
    
    ax2.set_ylabel('ATR', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_title(f'ATR ({period})', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制ATR图
plot_atr(df_with_atr.tail(100))

## 7. DMI/ADX指标

DMI衡量价格上涨和下跌的力度，ADX衡量趋势强度。

In [ ]:
def calculate_dmi_adx(df, period=14):
    """
    计算DMI和ADX指标
    
    参数:
        df: 股票数据DataFrame
        period: 周期，默认14
    
    返回:
        DataFrame，新增+di, -di, adx列
    """
    result = df.copy()
    
    # 计算上涨幅度和下跌幅度
    result['up_move'] = result['high'] - result['high'].shift(1)
    result['down_move'] = result['low'].shift(1) - result['low']
    
    # 计算+DM和-DM
    result['+dm'] = np.where((result['up_move'] > result['down_move']) & (result['up_move'] > 0), result['up_move'], 0)
    result['-dm'] = np.where((result['down_move'] > result['up_move']) & (result['down_move'] > 0), result['down_move'], 0)
    
    # 计算真实波幅（TR）
    high_low = result['high'] - result['low']
    high_close = abs(result['high'] - result['close'].shift(1))
    low_close = abs(result['low'] - result['close'].shift(1))
    result['tr'] = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    
    # 计算平滑的+DM、-DM和TR（使用Wilder平滑）
    result['+dm_smooth'] = result['+dm'].ewm(alpha=1/period, adjust=False).mean()
    result['-dm_smooth'] = result['-dm'].ewm(alpha=1/period, adjust=False).mean()
    result['tr_smooth'] = result['tr'].ewm(alpha=1/period, adjust=False).mean()
    
    # 计算+DI和-DI
    result['+di'] = 100 * result['+dm_smooth'] / result['tr_smooth']
    result['-di'] = 100 * result['-dm_smooth'] / result['tr_smooth']
    
    # 计算DX
    result['dx'] = 100 * abs(result['+di'] - result['-di']) / (result['+di'] + result['-di'])
    
    # 计算ADX（DX的MA）
    result[f'adx{period}'] = result['dx'].rolling(window=period).mean()
    
    return result

# 示例：计算DMI/ADX
df_with_dmi = calculate_dmi_adx(df_with_atr)
df_with_dmi[['trade_date', 'close', '+di', '-di', 'adx14']].tail(10)

In [ ]:
def plot_dmi_adx(df, period=14):
    """
    绘制DMI/ADX指标图
    
    参数:
        df: 包含DMI/ADX列的DataFrame
        period: 周期
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 上图：收盘价
    ax1.plot(df['trade_date'], df['close'], color='black', linewidth=1)
    ax1.set_ylabel('价格', fontsize=12)
    ax1.set_title('价格走势', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 下图：DMI/ADX
    ax2.plot(df['trade_date'], df['+di'], color='green', linewidth=1.5, label='+DI')
    ax2.plot(df['trade_date'], df['-di'], color='red', linewidth=1.5, label='-DI')
    
    col_name = f'adx{period}'
    ax2.plot(df['trade_date'], df[col_name], color='blue', linewidth=1.5, label='ADX')
    
    # 添加趋势强度参考线
    ax2.axhline(y=25, color='gray', linestyle='--', alpha=0.5)
    ax2.axhline(y=50, color='gray', linestyle=':', alpha=0.5)
    
    ax2.set_ylabel('DMI/ADX', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_title(f'DMI/ADX ({period})', fontsize=14, fontweight='bold')
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 100)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制DMI/ADX图
plot_dmi_adx(df_with_dmi.tail(100))

## 8. KDJ指标

KDJ是随机指标的改进版，包含K线、D线和J线，用于识别超买超卖和反转信号。

In [ ]:
def calculate_kdj(df, n=9, m1=3, m2=3):
    """
    计算KDJ指标
    
    参数:
        df: 股票数据DataFrame
        n: RSV周期，默认9
        m1: K值平滑参数，默认3
        m2: D值平滑参数，默认3
    
    返回:
        DataFrame，新增k, d, j列
    """
    result = df.copy()
    
    # 计算RSV（未成熟随机值）
    low_n = result['low'].rolling(window=n).min()
    high_n = result['high'].rolling(window=n).max()
    result['rsv'] = 100 * (result['close'] - low_n) / (high_n - low_n)
    
    # 计算K值和D值（使用平滑处理）
    result['k'] = result['rsv'].ewm(com=m1-1, adjust=False).mean()
    result['d'] = result['k'].ewm(com=m2-1, adjust=False).mean()
    
    # 计算J值
    result['j'] = 3 * result['k'] - 2 * result['d']
    
    return result

# 示例：计算KDJ
df_with_kdj = calculate_kdj(df_with_dmi)
df_with_kdj[['trade_date', 'close', 'k', 'd', 'j']].tail(10)

In [ ]:
def plot_kdj(df, n=9, m1=3, m2=3, overbought=80, oversold=20):
    """
    绘制KDJ指标图
    
    参数:
        df: 包含KDJ列的DataFrame
        n: RSV周期
        m1: K值平滑参数
        m2: D值平滑参数
        overbought: 超买线，默认80
        oversold: 超卖线，默认20
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 上图：收盘价
    ax1.plot(df['trade_date'], df['close'], color='black', linewidth=1)
    ax1.set_ylabel('价格', fontsize=12)
    ax1.set_title('价格走势', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 下图：KDJ
    ax2.plot(df['trade_date'], df['k'], color='blue', linewidth=1.5, label='K')
    ax2.plot(df['trade_date'], df['d'], color='red', linewidth=1.5, label='D')
    ax2.plot(df['trade_date'], df['j'], color='green', linewidth=1.5, label='J')
    
    # 添加超买超卖线
    ax2.axhline(y=overbought, color='red', linestyle='--', alpha=0.5)
    ax2.axhline(y=oversold, color='green', linestyle='--', alpha=0.5)
    ax2.fill_between(df['trade_date'], overbought, 100, alpha=0.1, color='red')
    ax2.fill_between(df['trade_date'], 0, oversold, alpha=0.1, color='green')
    
    ax2.set_ylabel('KDJ', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_title(f'KDJ ({n},{m1},{m2})', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制KDJ图
plot_kdj(df_with_kdj.tail(100))

## 9. VWAP指标

VWAP是成交量加权平均价，用于衡量机构交易的平均成本。

In [ ]:
def calculate_vwap(df, reset_daily=True):
    """
    计算VWAP指标
    
    参数:
        df: 股票数据DataFrame
        reset_daily: 是否每日重置，默认True
    
    返回:
        DataFrame，新增vwap列
    """
    result = df.copy()
    
    # 计算典型价格
    result['typical_price'] = (result['high'] + result['low'] + result['close']) / 3
    
    # 计算累计成交额和累计成交量
    if reset_daily:
        # 按日重置
        result['date'] = result['trade_date'].dt.date
        result['cum_amount'] = result.groupby('date')['amount'].cumsum()
        result['cum_vol'] = result.groupby('date')['vol'].cumsum()
    else:
        # 不重置（从数据开始累计）
        result['cum_amount'] = result['amount'].cumsum()
        result['cum_vol'] = result['vol'].cumsum()
    
    # 计算VWAP
    result['vwap'] = result['cum_amount'] / result['cum_vol']
    
    return result

# 示例：计算VWAP（模拟数据需要添加日期列）
df_with_vwap = calculate_vwap(df_with_kdj, reset_daily=False)
df_with_vwap[['trade_date', 'close', 'vwap']].tail(10)

In [ ]:
def plot_vwap(df):
    """
    绘制VWAP指标（叠加在价格图上）
    
    参数:
        df: 包含VWAP列的DataFrame
    """
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # 绘制收盘价
    ax.plot(df['trade_date'], df['close'], color='black', linewidth=1.5, label='收盘价')
    
    # 绘制VWAP
    ax.plot(df['trade_date'], df['vwap'], color='blue', linewidth=1.5, linestyle='--', label='VWAP')
    
    # 填充价格高于/低于VWAP的区域
    ax.fill_between(df['trade_date'], df['close'], df['vwap'], 
                   where=(df['close'] >= df['vwap']), 
                   alpha=0.2, color='green', label='价格 > VWAP')
    ax.fill_between(df['trade_date'], df['close'], df['vwap'], 
                   where=(df['close'] < df['vwap']), 
                   alpha=0.2, color='red', label='价格 < VWAP')
    
    ax.set_title('VWAP (成交量加权平均价)', fontsize=14, fontweight='bold')
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('价格', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：绘制VWAP图
plot_vwap(df_with_vwap.tail(100))

## 10. 综合示例：参数调节效果对比

展示不同参数设置对技术指标的影响。

In [ ]:
def compare_rsi_parameters(df, periods=[7, 14, 21]):
    """
    对比不同RSI周期的效果
    
    参数:
        df: 股票数据DataFrame
        periods: 要对比的RSI周期列表
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # 上图：收盘价
    ax1.plot(df['trade_date'], df['close'], color='black', linewidth=1)
    ax1.set_ylabel('价格', fontsize=12)
    ax1.set_title('价格走势', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 下图：不同周期的RSI
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    for i, period in enumerate(periods):
        df_temp = calculate_rsi(df, period=period)
        ax2.plot(df_temp['trade_date'], df_temp[f'rsi{period}'], 
                color=colors[i % len(colors)], 
                linewidth=1.5, 
                label=f'RSI({period})')
    
    ax2.axhline(y=70, color='red', linestyle='--', alpha=0.5)
    ax2.axhline(y=30, color='green', linestyle='--', alpha=0.5)
    ax2.set_ylabel('RSI', fontsize=12)
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_title('RSI参数对比', fontsize=14, fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：对比不同RSI周期
compare_rsi_parameters(df.tail(100), periods=[7, 14, 21])

In [ ]:
def compare_macd_parameters(df, configs=[(12,26,9), (5,35,5), (8,17,9)]):
    """
    对比不同MACD参数的效果
    
    参数:
        df: 股票数据DataFrame
        configs: 要对比的MACD参数列表，每个元素为(fast, slow, signal)
    """
    n_configs = len(configs)
    fig, axes = plt.subplots(n_configs, 1, figsize=(14, 4*n_configs), sharex=True)
    
    if n_configs == 1:
        axes = [axes]
    
    for i, (fast, slow, signal) in enumerate(configs):
        ax = axes[i]
        
        # 计算MACD
        df_temp = calculate_macd(df, fast_period=fast, slow_period=slow, signal_period=signal)
        
        # 绘制MACD
        ax.plot(df_temp['trade_date'], df_temp['dif'], color='blue', linewidth=1.5, label='DIF')
        ax.plot(df_temp['trade_date'], df_temp['dea'], color='red', linewidth=1.5, label='DEA')
        
        colors = ['red' if x < 0 else 'green' for x in df_temp['macd']]
        ax.bar(df_temp['trade_date'], df_temp['macd'], color=colors, alpha=0.5)
        
        ax.set_title(f'MACD ({fast},{slow},{signal})', fontsize=12, fontweight='bold')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 示例：对比不同MACD参数
compare_macd_parameters(df.tail(100), configs=[(12,26,9), (5,35,5), (8,17,9)])

## 总结

本Notebook提供了所有技术指标的计算代码和可视化示例。您可以通过调节参数来观察指标的变化，深入理解每个技术指标的特性。

### 关键要点：
1. **MA**：趋势识别，周期越短越敏感
2. **RSI**：超买超卖，周期越短越敏感
3. **MACD**：趋势反转，参数影响快慢线灵敏度
4. **布林带**：波动范围，标准差倍数影响通道宽度
5. **ATR**：波动强度，用于止损设置
6. **DMI/ADX**：趋势强度，ADX越高趋势越强
7. **KDJ**：超买超卖，对价格变化更敏感
8. **VWAP**：机构成本，价格与VWAP的关系反映多空力量

### 下一步：
- 尝试将多个指标组合使用（如RSI + MACD）
- 编写自动交易信号识别代码
- 将代码集成到完整的交易系统中